# Notebook for plotting canonical evolution Movie S1 to accompany Figure 6 
## Preamble
### Load packages required

In [8]:
#SJL 7/2020
#Script to plot the change in surface length for planets for an example tidal evolution
#plots on a 2D map showing the change in shape and surface


###########################################################
###########################################################
###########################################################
import numpy as np
import scipy as sp
import sys
import os
import struct
from scipy import constants as const

from scipy.signal import savgol_filter

#package to use wildcards 
import fnmatch

import csv

from scipy.interpolate import LinearNDInterpolator
from scipy.interpolate import griddata

#plotting packages
import matplotlib as mpl
import matplotlib.pyplot as plt
import pylab
import matplotlib.cm as cm
from matplotlib import gridspec


cwd = os.getcwd()
print(cwd)
if sys.platform== 'darwin':
    sys.path.insert(0, cwd+'/Support_scipts')
    print(cwd+'/Support_scripts')
elif (sys.platform== 'win32') | (sys.platform== 'win64'):
    sys.path.insert(0,cwd+"\\Support_scipts")
    
#HERCULES_structures
from HERCULES_structures import *
from surface_size_calc import *
from HERCULES_random_planet_database_structure_1D import *

#functions for calculating non-evenly spaced numerical differentials
from gradients import *

#import colormaps
import colormaps as cmaps
import matplotlib.cm as cm

import svglib.svglib as svglib
svglib.register_font('helvetica', './Helvetica.ttc')

/Users/vq21447/Documents/Lock_2026_SI
/Users/vq21447/Documents/Lock_2026_SI/Support_scripts


('helvetica', True)

### Define constants

In [ ]:
########################################################################################
########################################################################################
########################################################################################
#CONSTANTS
MEarth=5.972E24
LEM=3.5E34
REarth=6.371E6
MMoon=7.34767309E22

aMoon=0.3844E9
aCassini=30*REarth
aRoche=2.9*REarth

#for HERCULES
MEarth_H=5.9879648E24
LEM_H=3.53E34

### Set parameters and scenario to plot

In [ ]:
########################################################################################
########################################################################################
########################################################################################
#PARAMS
#info for HERCULES arrays
Hdir='Earth_correct_params_S3.20c'
Hname='Earth_correct_params_S3.20c'


#Which combination of panels to plot
#Single shape plots
#10: Latitudinal
#11: Longitudinal
#12: Area
#Single shape plots with semi-major axis and AM
#20: Latitudinal
#21: Longitudinal
#22: Area
#Animated version of full stack
#3: Column
#Single shape plots with semi-major axis, AM, and integrated deformation
#41: Longitudinal
#4: Square
flag_plot=20

#number of contor points
Ncont=1000 #500

#time step for plotting
tstep_plot=0.05 #Myr

#constants for
k2Q=0.3/100.0
a0=2.9*REarth
Ltot=LEM

#time to plot until
tmax=20 #Myr

#output for snapshots
output_dir='Movie_slides/MovieS1_slides'

if os.path.isdir(output_dir)==False:
    os.mkdir(output_dir)

## Main script
### Read in HERCULES database

In [ ]:
########################################################################################
########################################################################################
########################################################################################
#MAIN

########################################################################################
#read in the database

Hdatabase=HERCULES_random_planet_database_1D()
Hdatabase.make_array(Hdir,Hname)
Hdatabase.initialize_interpolation([0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1],\
                                   [0,0,[0],0,0,0,0,[0],[0],[0],[0],[0],[0],[0],[0],[0],[0],[0]], flag_extrap=1)

#extract the latitudes for each Mu point
Nmu=Hdatabase.parr[0].Nmu
lat=np.arccos(Hdatabase.parr[0].layers[0].mu)*180/np.pi

### Calculate the orbital evolution

In [12]:
#Find the time, AM and a to plot
time=np.arange(0.0,tmax*1E6,tstep_plot*1E6)*const.year
Nt_plot=np.size(time) #number of time points to plot

#calculate the orbit
kappa=3.0*k2Q*MMoon/MEarth*(REarth**5)*np.sqrt(const.G*(MEarth+MMoon))
a=((13.0/2.0)*(kappa*time+(2.0/13.0)*(a0**(13.0/2.0))))**(2.0/13.0)
L=Ltot-(MMoon)*np.sqrt(const.G*a*(MEarth+MMoon))
dLdt=-0.5*MMoon/np.sqrt(const.G*a*(MEarth+MMoon))*const.G*(MEarth+MMoon)*kappa*(a**(-11.0/2.0))



### Calculate the corresponding changes in length

In [13]:
#now run through all time steps and calculate the change in length
print('begin')

checkpoints=np.linspace(1,101,101)

#run through and extract lengths and areas at each time point
dl_lat=np.zeros((Nt_plot,Nmu))
dl_lon=np.zeros((Nt_plot,Nmu))
dA=np.zeros((Nt_plot,Nmu))

ddl_lat_dL=np.zeros((Nt_plot,Nmu))
ddl_lon_dL=np.zeros((Nt_plot,Nmu))
ddA_dL=np.zeros((Nt_plot,Nmu))

ddl_lat_dt=np.zeros((Nt_plot,Nmu))
ddl_lon_dt=np.zeros((Nt_plot,Nmu))
ddA_dt=np.zeros((Nt_plot,Nmu))

ddl_lon_dt_max=np.zeros(Nt_plot)
ddl_lon_dt_min=np.zeros(Nt_plot)
ddl_lat_dt_max=np.zeros(Nt_plot)
ddl_lat_dt_min=np.zeros(Nt_plot)

rsurf=np.zeros((Nt_plot,Nmu))

dLdt=gradient2(time,L)

count=-1
for i in np.arange(Nt_plot):
    count+=1
    #print(i, np.size(steps),count)
    if (i*1.0/Nt_plot*100)>checkpoints[0]:
        print(i*1.0/Nt_plot*100, '%')
        checkpoints=checkpoints[1:]
    
    temp_data=Hdatabase.interp_database(L[i],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1],\
                                       [0,0,[0],0,0,0,0,[0],[0],[0],[0],[0],[0],[0],[0],[0],[0],[0]],flag_extrap=1)

    temp=np.asarray(temp_data[16])

    dl_lat[count,:]=temp[:,0]
    dl_lon[count,:]=temp[:,1]
    dA[count,:]=temp[:,2]
    
    ddl_lat_dL[count,:]=temp[:,3]
    ddl_lon_dL[count,:]=temp[:,4]
    ddA_dL[count,:]=temp[:,5]
    
    ddl_lat_dt[count,:]=temp[:,3]*dLdt[count]
    ddl_lon_dt[count,:]=temp[:,4]*dLdt[count]
    ddA_dt[count,:]=temp[:,5]*dLdt[count]
    
    ddl_lon_dt_max[count]=np.amax(temp[:,4]*dLdt[count])
    ddl_lon_dt_min[count]=np.amin(temp[:,4]*dLdt[count])
    
        
    rsurf[i,:]=np.asarray(temp_data[17])
    
    
    ## integrated deformation
    #longitudinal is easy. Accomodated around minor circle
    ddl_lon_dt_max[count]=np.amax(ddl_lon_dt[count,:])*const.year*2*np.pi
    ddl_lon_dt_min[count]=np.amin(ddl_lon_dt[count,:])*const.year*2*np.pi
    
    #latitudinal is harder. Need to integrate over surface
    temp=np.where(ddl_lat_dt[count,:]<0)[0][-1]
    
    #print(lat[0:temp])
    ddl_lat_dt_min[count]=2*integrate.trapz(ddl_lat_dt[count,0:temp]*const.year*np.pi/180.0,90.0-lat[0:temp])
    ddl_lat_dt_max[count]=integrate.trapz(ddl_lat_dt[count,(temp+1):]*const.year*np.pi/180.0,90.0-lat[(temp+1):])

print('end')

begin
1.25 %
2.25 %
3.25 %
4.25 %
5.25 %
6.25 %
7.000000000000001 %
8.25 %
9.25 %
10.25 %
11.25 %
12.25 %
13.25 %
14.000000000000002 %
15.25 %
16.25 %
17.25 %
18.25 %
19.25 %
20.25 %
21.25 %
22.25 %
23.25 %
24.25 %
25.25 %
26.25 %
27.250000000000004 %
28.000000000000004 %
29.25 %
30.25 %
31.25 %
32.25 %
33.25 %
34.25 %
35.25 %
36.25 %
37.25 %
38.25 %
39.25 %
40.25 %
41.25 %
42.25 %
43.25 %
44.25 %
45.25 %
46.25 %
47.25 %
48.25 %
49.25 %
50.24999999999999 %
51.24999999999999 %
52.25 %
53.25 %
54.25 %
55.00000000000001 %
56.00000000000001 %
57.25 %
58.25 %
59.25 %
60.25 %
61.25000000000001 %
62.25000000000001 %
63.24999999999999 %
64.25 %
65.25 %
66.25 %
67.25 %
68.25 %
69.25 %
70.25 %
71.25 %
72.25 %
73.25 %
74.25 %
75.25 %
76.25 %
77.25 %
78.25 %
79.25 %
80.25 %
81.25 %
82.25 %
83.25 %
84.25 %
85.25 %
86.25 %
87.25 %
88.25 %
89.25 %
90.25 %
91.25 %
92.25 %
93.25 %
94.25 %
95.25 %
96.25 %
97.25 %
98.25 %
99.25 %
end


### Run through and plot each slide

In [14]:

print('begin')

#################################################
#version with a plot of AM and semi-major axis and latitudinal deformation
#now in a square arangement
#################################################

boxlim=18
z_plot=np.linspace(-boxlim/2, boxlim/2, num=Ncont)
x_plot=np.linspace(-boxlim/2, boxlim/2, num=Ncont)

XXX, ZZZ = np.meshgrid(x_plot, z_plot)

#do the first one twice to make sure the graphics are working
temp=np.append(0,np.arange(np.size(time))[0:])
for k in temp[0:]:
    print(k+1, str(time[k]/1E6)+' Myr', str((k+1)/Nt_plot*100)+'%')
    #loop over all points on the surface
    Nrow_data=Ncont
    x_data=np.zeros(Hdatabase.parr[0].Nmu*Nrow_data*2)
    z_data=np.zeros(Hdatabase.parr[0].Nmu*Nrow_data*2)
    ddl_lon_dt_data=np.zeros(Hdatabase.parr[0].Nmu*Nrow_data*2)
    

    for i in np.arange(Hdatabase.parr[0].Nmu):
        x=rsurf[k,i]*np.sin(np.arccos(Hdatabase.parr[0].mu[i]))
        z=rsurf[k,i]*Hdatabase.parr[0].mu[i]
        x_data[2*i*Nrow_data:(i+1)*2*Nrow_data]=np.append(np.linspace(-x, x, Nrow_data),np.linspace(-x, x, Nrow_data))
        z_data[2*i*Nrow_data:(i+1)*2*Nrow_data]=np.append(np.ones(Nrow_data)*z,-np.ones(Nrow_data)*z)
        ddl_lon_dt_data[2*i*Nrow_data:(i+1)*2*Nrow_data]=np.ones(2*Nrow_data)*ddl_lon_dt[k,i]
        
    DLON=griddata(((x_data.flatten()/1E6,z_data.flatten()/1E6)),ddl_lon_dt_data.flatten()*const.year*np.pi/180.0,(XXX,ZZZ),fill_value=np.nan)#, method='linear')
    
    #initialise the figure
    fig = plt.figure(figsize=(7.7,5.5))
    gs0 = gridspec.GridSpec(1, 2, width_ratios=[0.9,1])
    
    gs00 = gridspec.GridSpecFromSubplotSpec(3, 2,
                                            width_ratios=[1,0.05],
                                            height_ratios=[1,1,1.65],
                                    subplot_spec=gs0[0])
    gs01 = gridspec.GridSpecFromSubplotSpec(3, 2,
                                            width_ratios=[1,0.05],
                                            height_ratios=[0.5,2,0.5],
                                    subplot_spec=gs0[1])

    
    ax=[[]]
    ax[0].append(plt.subplot(gs00[0]))
    ax[0].append(plt.subplot(gs00[2], sharex=ax[0][0]))
    ax[0].append(plt.subplot(gs01[2]))
    ax[0].append(plt.subplot(gs00[4], sharex=ax[0][0]))
#         ax[0].append(plt.subplot(gs02[0]))

    ax_col=[[]]
    ax_col[0].append(plt.subplot(gs01[3]))
    
    
    font = {
    'family' : 'Helvetica',
            'weight' : 'normal',
            'size'   : 8}
    mpl.rc('font', **font)
    
    col=cmaps.parula([0.15,0.85])
    

    
    ax[0][0].plot(time[0:k]/const.year/1E6, a[0:k]/REarth, 'k-', linewidth=1.5)

    ax[0][1].plot(time[0:k]/const.year/1E6, L[0:k]/LEM, 'k-', linewidth=1.5)
    
    ax[0][0].set_ylim([1.5,24])
    ax[0][1].set_ylim([0.46,0.85])
    
    ax[0][0].set_xlim([-0.05*time[-1]/const.year/1E6,1.05*time[-1]/const.year/1E6])
    ax[0][1].set_xlim([-0.05*time[-1]/const.year/1E6,1.05*time[-1]/const.year/1E6])
    ax[0][3].set_xlim([-0.05*time[-1]/const.year/1E6,1.05*time[-1]/const.year/1E6])
    
    if k==Nt_plot-1:
        ind_max=-1
        ax[0][3].plot([0,time[ind_max]/const.year/1E6],[60E-3,60E-3],':', color=col[1],linewidth=1.0, label='Average subduction') #avererage subduction
        ax[0][3].plot([0,time[ind_max]/const.year/1E6],[50E-3,50E-3],':', color=col[0],linewidth=1.0, label='Average ridge') #average mid-ocean ridge
        ax[0][3].plot([0,time[ind_max]/const.year/1E6],[15E-3,15E-3],'--', color=col[1],linewidth=1.0, label='Slow subduction') #slow subduction
        ax[0][3].plot([0,time[ind_max]/const.year/1E6],[8E-3,8E-3],'--', color=col[0],linewidth=1.0, label='Slow ridge') #slow mid ocean ridge

    
    ax[0][3].plot(time[0:k]/const.year/1E6, np.absolute(ddl_lon_dt_max[0:k]), '-', color=col[0], linewidth=1.5)
    ax[0][3].plot(time[0:k]/const.year/1E6, np.absolute(ddl_lon_dt_min[0:k]), '-', color=col[1], linewidth=1.5)
    
    ax[0][3].set_yscale('log')
    ax[0][3].set_ylim([1E-3,1.9E1])
    
    
    lat_contours=np.linspace(-7,-2.5,18+1)
    lon_contours=np.linspace(-7,-2.5,18+1)
    Acontours=np.linspace(-2,3,20+1)

    lat_labels=np.asarray([-7,-6,-5,-4,-3])
    lon_labels=np.asarray([-7,-6,-5,-4,-3])
    Alabels=np.asarray([-2,-1,0,1,2,3])
    
    contour2 = ax[0][2].contourf(XXX, ZZZ, np.log10(np.absolute(DLON)), levels=lat_contours, colors=cmaps.parula_r(np.linspace(1, 0, np.size(lat_contours)-1)))
    contour20 = ax[0][2].contour(XXX, ZZZ, DLON, linewidths=0.5, levels=[0.0], colors=['k'], linestyles='solid')
       
        
    for c in contour2.collections:
        c.set_rasterized(True)
        
    #plot the intial outline
    ax[0][2].plot(rsurf[0,:]*np.sin(np.arccos(Hdatabase.parr[0].mu))/1E6,rsurf[0,:]*Hdatabase.parr[0].mu/1E6, '--', color='k', linewidth=1.5)
    ax[0][2].plot(-rsurf[0,:]*np.sin(np.arccos(Hdatabase.parr[0].mu))/1E6,rsurf[0,:]*Hdatabase.parr[0].mu/1E6, '--', color='k', linewidth=1.5)
    ax[0][2].plot(rsurf[0,:]*np.sin(np.arccos(Hdatabase.parr[0].mu))/1E6,-rsurf[0,:]*Hdatabase.parr[0].mu/1E6, '--', color='k', linewidth=1.5)
    ax[0][2].plot(-rsurf[0,:]*np.sin(np.arccos(Hdatabase.parr[0].mu))/1E6,-rsurf[0,:]*Hdatabase.parr[0].mu/1E6, '--', color='k', linewidth=1.5)

        
    cbar2=plt.colorbar(contour2, cax=ax_col[0][0], orientation='vertical',ticks=lon_labels)
    
    cbar2.set_label(r'Rate [$\log_{10} (|$m deg$^{-1}$ yr$^{-1}$$|)$]', labelpad=-40)
    
    ax[0][2].set_aspect('equal', adjustable='box')
    
    
    #label axis
    ax[0][0].text(0.04, 0.93, 'A: '+"{:.2f}".format(round(time[k]/const.year/1E6,2))+' Myrs', horizontalalignment='left',verticalalignment='top', fontsize=10,transform=ax[0][0].transAxes, color='k')
    ax[0][1].text(0.04, 0.07, 'B', horizontalalignment='left',verticalalignment='bottom', fontsize=10,transform=ax[0][1].transAxes, color='k')
    ax[0][2].text(0.04, 0.96, 'D: Longitudinal', horizontalalignment='left',verticalalignment='top', fontsize=10,transform=ax[0][2].transAxes, color='k')
    ax[0][3].text(0.04, 0.04, 'C: Longitudinal', horizontalalignment='left',verticalalignment='bottom', fontsize=10,transform=ax[0][3].transAxes, color='k')
    

    ax[0][2].set_xlabel('[10$^6$ m]')
    
    ax[0][2].set_ylabel('[10$^6$ m]')
    
    ax[0][0].set_ylabel(r"$a_{\rm Moon}$ [$R_{\rm Earth}$]")
    ax[0][1].set_ylabel(r"AM Earth [$L_{\rm EM}$]")
    ax[0][3].set_ylabel(r"Int. deform. rate [m yr$^{-1}$]")
    ax[0][3].set_xlabel(r"Time [Myrs]")
    
    plt.setp( ax[0][0].get_xticklabels(), visible=False)
    plt.setp( ax[0][1].get_xticklabels(), visible=False)

    for i in np.arange(4):
        ax[0][i].tick_params(direction="in", top=True, right=True)


    ax_col[0][0].tick_params(direction="in")
    
    if k==Nt_plot-1:
        ax[0][3].legend(frameon=False, handlelength=1.9, loc='upper right')

    fig.tight_layout()
    plt.savefig(output_dir+'/MovieS1_slide_'+str(k+1).zfill(5)+'.png', dpi=600, format='png')
    
    plt.close(fig)

print('done')


begin
1 0.0 Myr 0.25%


/var/folders/mm/ct8d2r3j48bcp_kyghqh_n7r0000gq/T/ipykernel_39429/294580607.py:107: MatplotlibDeprecationWarning: The collections attribute was deprecated in Matplotlib 3.8 and will be removed two minor releases later.
  for c in contour2.collections:


1 0.0 Myr 0.25%
2 1576800.0 Myr 0.5%
3 3153600.0 Myr 0.75%
4 4730400.0 Myr 1.0%
5 6307200.0 Myr 1.25%
6 7884000.0 Myr 1.5%
7 9460800.0 Myr 1.7500000000000002%
8 11037600.0 Myr 2.0%
9 12614400.0 Myr 2.25%
done
